# Implementation of kNN Failure Splitting

kNN Failure Splitting is one of the four suggested biased splitting methods in this project. It is *suggested* merely as an example of what is possible when we think in terms of answering a chemically meaningful question by constructing biased train test split to validate model performance.

**The question we want to answer using kNN Failure** is `"How does the model performance change when increasing fractions of test molecule's k nearest training neighbours disagree with their true activity?"`

> *Note: This is equivalent to Activity Cliff Splitting that we constructed before. Activity Cliff can be considered a 1NN disagreement when the 1NN similarity is greter than some threshold and disagreement is greater than activity threshold.*

All the questions tend to follow the schema: "How does the model performance change when `<increasing/decreasing>` fractions of test molecules `<question>`".
We can get that answer with relative ease using the term `Bias`. Bias (bounded [0,1]) here means the fraction of test set that answers True for the given chemically meaningful question.

We want a train/test split where *bias* fraction of the test set "fails" for a kNN trained on the train set. A kNN failure here means: the k nearest training neighbours of the test molecule have a mean activity that disagrees with the molecule's true activity by at least some threshold.

The bias parameter is the fraction of test molecules such that kNN on training data will fail on them. By sweeping the bias from 0 to 1 and benchmarking models at each setting, we characterise how prediction changes as the test set gets progressively harder.

Throughout the rest of this notebook, "kNN" means k nearest neighbours **inside the training set**, not in the full dataset. The full-dataset notion shows up only as a construction trick.

Effective bias is re-evaluated after the split is frozen: for each test molecule find its k nearest training neighbours, compute consensus, check disagreement. Test molecules with fewer than k qualifying training neighbours are excluded from the count.

In [1]:
import os
import tempfile

import numpy as np
from PIL import Image

from molecularnetwork import (
    smiles_to_ecfp4_bitvect,
    compute_similarity_matrix,
    molecular_network_from_list,
    visualise_molnet_split,
)

In [2]:
import pandas as pd

In [3]:
UNASSIGNED_NODE = 0
TRAIN_NODE = 1
TEST_NODE = 2

A candidate failure is a molecule M whose top-k most similar molecules in the **full dataset** (above the similarity threshold) have a mean activity that disagrees with M by at least the activity threshold.

For each molecule we also record its k neighbour indices, because we'll need them in the next step.

In [12]:
def find_failure_n_edges(similarity_matrix, activity_values, similarity_threshold, activity_threshold, n_neighbors):
    n_molecules = len(activity_values)
    n_edges = []
    for molecule_index in range(n_molecules):
        similarities = similarity_matrix[molecule_index].copy()
        similarities[molecule_index] = -1.0
        qualifying = np.where(similarities >= similarity_threshold)[0]
        if len(qualifying) < n_neighbors:
            continue
        top_k = qualifying[np.argsort(similarities[qualifying])[::-1][:n_neighbors]]
        consensus = float(activity_values[top_k].mean())
        disagreement = abs(consensus - float(activity_values[molecule_index]))
        if disagreement >= activity_threshold:
            n_edges.append((int(molecule_index), tuple(int(n) for n in top_k)))
    return n_edges

We now have a list of (molecule, k-neighbours) tuples. To make the split:

1. Put the molecule in test.
2. Put all k neighbours in train.

The invariant this preserves: the k molecules we put in train are by construction the k *most similar to M anywhere in the dataset*, so once they're in train, nothing else is more similar to M, so they are also M's k nearest training neighbours. The failure verdict computed pre-split therefore equals the failure verdict computed post-split. No re-evaluation needed for placed candidates.

To keep this invariant intact, the walk uses two skip rules:

- If M was already placed in train (as someone else's neighbour), don't put it in test.
- If any of M's neighbours was already placed in test, skip these edges entirely, because we no longer control M's training neighbourhood cleanly.

We walk in random order (shuffled by the local rng) so the bias parameter samples representatively from the candidate set rather than always picking the worst failures first. We stop when we've placed the target number of failures.

In [14]:
def walk_failure_n_edges(failure_n_edges, n_molecules, n_failure_test_target):
    assignment = np.full(n_molecules, UNASSIGNED_NODE, dtype=np.int8)
    n_failures_placed = 0
    for molecule_index, neighbor_indices in failure_n_edges:
        if n_failures_placed >= n_failure_test_target:
            break
        if assignment[molecule_index] == TRAIN_NODE:
            continue
        if any(assignment[n] == TEST_NODE for n in neighbor_indices):
            continue
        assignment[molecule_index] = TEST_NODE
        for neighbor_index in neighbor_indices:
            assignment[neighbor_index] = TRAIN_NODE
        n_failures_placed += 1
    return assignment

$\color{red}CAUTION$

Each placed failure consumes 1 test slot and k train slots (in the worst case, with no neighbour-set overlap). With test_fraction f, the maximum bias we can reach is bounded by the train budget:

$$ b_{\max} \leq \frac{1 - f}{k \cdot f} $$

For f = 0.2 and k = 3 this is 1.33, so bias = 1.0 is reachable. For k = 5 it drops to 0.80, and for k = 10 to 0.40. In a clustered dataset the ceiling rises above this worst-case bound (neighbour sets overlap, so placed failures effectively share train slots), but the bound itself is dataset-independent and worth reporting.

After the walk, the test set typically has fewer molecules than `target_test_size`. We need to fill the remaining slots from unassigned molecules.
The naïve choice (pick randomly from anything unassigned) leaks bias upward: leftover candidate molecules picked at random for test can become accidental failures, pushing effective_bias above intended_bias.
We will prefer non-candidates for the fill first, only using the candidate pool when non-candidates run out

Once the partition is fixed, we measure what the realised bias actually is. This uses the **training-set kNN** definition: for each test molecule, find its top k similar training neighbours, compute their mean activity, check disagreement against the molecule's own activity. For placed candidates the invariant guarantees they evaluate as failures. For randomly-filled test molecules, the evaluation tells us what happened. Test molecules with fewer than k qualifying training neighbours can't be evaluated (kNN itself isn't defined for them). We exclude these from the count by marking them NaN. The effective bias is the mean of the non-NaN results. Obviously setting the similarity threshold to 0 will make this behave as a real kNN shall perform on these test compounds.

In [15]:
def evaluate_knn_failure_question(test_indices, train_indices, activity_values, similarity_matrix, similarity_threshold, activity_threshold, n_neighbors):
    results = np.full(len(test_indices), np.nan, dtype=float)
    if len(test_indices) == 0 or len(train_indices) == 0:
        return results
    for position, test_idx in enumerate(test_indices):
        similarities_to_train = similarity_matrix[test_idx][train_indices]
        qualifying = np.where(similarities_to_train >= similarity_threshold)[0]
        if len(qualifying) < n_neighbors:
            continue
        top_k_positions = qualifying[np.argsort(similarities_to_train[qualifying])[::-1][:n_neighbors]]
        top_k_train_indices = train_indices[top_k_positions]
        consensus = float(activity_values[top_k_train_indices].mean())
        disagreement = abs(consensus - float(activity_values[test_idx]))
        results[position] = 1.0 if disagreement >= activity_threshold else 0.0
    return results

In [16]:
def effective_bias_from_question_results(question_results):
    if question_results.size == 0:
        return 0.0
    evaluable = question_results[~np.isnan(question_results)]
    if evaluable.size == 0:
        return 0.0
    return float(evaluable.mean())

Let's make this into a class and get the splitter 

In [17]:
class KNNFailureSplitter:
    def __init__(self, similarity_threshold, activity_threshold, n_neighbors, test_fraction=0.2):
        self.similarity_threshold = similarity_threshold
        self.activity_threshold = activity_threshold
        self.n_neighbors = n_neighbors
        self.test_fraction = test_fraction

    def split_for_intended_bias(self, smiless, activity_values, intended_bias, random_seed):
        if not (0.0 <= intended_bias <= 1.0):
            raise ValueError(f"intended_bias must be in [0, 1], got {intended_bias}")

        rng = np.random.default_rng(random_seed)
        n_molecules = len(smiless)
        target_test_size = int(self.test_fraction * n_molecules)
        n_failure_test_target = int(intended_bias * target_test_size)

        fps_bitvect = [smiles_to_ecfp4_bitvect(s) for s in smiless]
        similarity_matrix = compute_similarity_matrix(fps_bitvect)

        failure_n_edges = find_failure_n_edges(
            similarity_matrix, activity_values,
            self.similarity_threshold, self.activity_threshold, self.n_neighbors,
        )
        shuffled_order = rng.permutation(len(failure_n_edges))
        failure_n_edges = [failure_n_edges[i] for i in shuffled_order]

        assignment = walk_failure_n_edges(failure_n_edges, n_molecules, n_failure_test_target)

        candidate_set = {molecule_index for molecule_index, _ in failure_n_edges}
        is_candidate_mask = np.zeros(n_molecules, dtype=bool)
        if candidate_set:
            is_candidate_mask[list(candidate_set)] = True

        unassigned_indices = np.where(assignment == UNASSIGNED_NODE)[0]
        unassigned_non_candidate_indices = unassigned_indices[~is_candidate_mask[unassigned_indices]]
        unassigned_candidate_indices = unassigned_indices[is_candidate_mask[unassigned_indices]]

        n_random_fill = target_test_size - int((assignment == TEST_NODE).sum())
        if n_random_fill > 0:
            if len(unassigned_non_candidate_indices) >= n_random_fill:
                random_test_indices = rng.choice(unassigned_non_candidate_indices, size=n_random_fill, replace=False)
            else:
                shortfall = n_random_fill - len(unassigned_non_candidate_indices)
                candidate_topup_indices = rng.choice(
                    unassigned_candidate_indices,
                    size=min(shortfall, len(unassigned_candidate_indices)),
                    replace=False,
                )
                random_test_indices = np.concatenate([unassigned_non_candidate_indices, candidate_topup_indices])
            assignment[random_test_indices] = TEST_NODE

        assignment[assignment == UNASSIGNED_NODE] = TRAIN_NODE

        train_indices = np.where(assignment == TRAIN_NODE)[0]
        test_indices = np.where(assignment == TEST_NODE)[0]

        question_results = evaluate_knn_failure_question(
            test_indices, train_indices,
            np.asarray(activity_values, dtype=float),
            similarity_matrix, self.similarity_threshold, self.activity_threshold, self.n_neighbors,
        )
        effective_bias = effective_bias_from_question_results(question_results)

        return train_indices, test_indices, effective_bias

    def split(self, smiless, activity_values, intended_biases, n_repeats):
        for intended_bias in intended_biases:
            for repeat_index in range(n_repeats):
                train_indices, test_indices, effective_bias = self.split_for_intended_bias(
                    smiless=smiless, activity_values=activity_values,
                    intended_bias=intended_bias, random_seed=repeat_index,
                )
                yield train_indices, test_indices, effective_bias, intended_bias, repeat_index

    def visualise_splits(self, smiless, activity_values, intended_biases, n_repeats, output_path, duration=500):
        G = molecular_network_from_list(smiless, activity_values, self.similarity_threshold, self.activity_threshold)
        with tempfile.TemporaryDirectory() as tmpdir:
            paths = []
            for frame_index, (train_idx, test_idx, effective_bias, intended_bias, _) in enumerate(
                self.split(smiless, activity_values, intended_biases, n_repeats)
            ):
                p = os.path.join(tmpdir, f"frame_{frame_index:04d}.png")
                visualise_molnet_split(G, train_idx, test_idx, effective_bias, intended_bias, filepath=p)
                paths.append(p)
            frames = [Image.open(p) for p in paths]
            frames[0].save(output_path, save_all=True, append_images=frames[1:], duration=duration, loop=0)

Let's see if this works. We can check `n_neighbors` to see how the structural ceiling kicks in. For k = 1 this should produce splits essentially identical to the 1-NN cliff splitter (consensus of one molecule equals that molecule's activity, so failure ≡ cliff)

In [32]:
df = pd.read_csv("../data/standardized/target_CHEMBL325-1.IC50.csv")

smiless = df['standardized_smiles'].values
activity_values = df['pchembl_value'].values

fps = [smiles_to_ecfp4_bitvect(smiles) for smiles in smiless]
similarity_matrix = compute_similarity_matrix(fps)

fail_edges = find_failure_edges(similarity_matrix, activity_values, similarity_threshold=0.7, activity_threshold=0.5, n_neighbors=3)
print(f"Failure Edges = {len(fail_edges)} | Test set size = {int(len(smiless)*0.2)}")

Failure Edges = 241 | Test set size = 269


In [30]:
knn_failure_splitter = KNNFailureSplitter(
    similarity_threshold=0.7,
    activity_threshold=0.5,
    n_neighbors=3,
    test_fraction=0.2,
)

In [31]:
knn_failure_splitter.visualise_splits(
    df['standardized_smiles'].values,
    df['pchembl_value'].values,
    intended_biases=np.arange(0, 11) / 10,
    n_repeats=1,
    output_path="knn_failure_sweep.gif",
)

![kNN Failure Split sweep](./knn_failure_sweep.gif)

In [34]:
df

,assay_chembl_id,compound_chembl_id,canonical_smiles,pchembl_value,standardized_smiles
0,CHEMBL695136,CHEMBL317718,Nc1ccccc1NC(=O)CCCCCCC(=O)c1cccc(-c2ccccc2)c1,6.00,Nc1ccccc1NC(=O)CCCCCCC(=O)c1cccc(-c2ccccc2)c1
1,CHEMBL695136,CHEMBL420998,Nc1ccccc1NC(=O)CCCCCCC(=O)c1ccc(-c2cccnc2)cc1,5.00,Nc1ccccc1NC(=O)CCCCCCC(=O)c1ccc(-c2cccnc2)cc1
2,CHEMBL695136,CHEMBL95792,Nc1ccccc1NC(=O)CCCCCCC(=O)c1ccc(-c2ccccc2)nc1,5.52,Nc1ccccc1NC(=O)CCCCCCC(=O)c1ccc(-c2ccccc2)nc1
3,CHEMBL695136,CHEMBL318613,Nc1ccccc1NC(=O)CCCCCn1cnc2ccccc2c1=O,4.92,Nc1ccccc1NC(=O)CCCCCn1cnc2ccccc2c1=O
4,CHEMBL695136,CHEMBL329745,Nc1ccccc1NC(=O)CCCCCN1C(=O)c2cccc3c(CN4CCCCC4)...,5.70,Nc1ccccc1NC(=O)CCCCCn1c(O)c2cc(O)c(=CN3CCCCC3)...
...,...,...,...,...,...
1343,CHEMBL5047887,CHEMBL5081173,Nc1ccccc1NC(=O)c1ccc(CSc2nc(NC3CCNCC3)c3cccn3n...,6.31,Nc1ccccc1NC(=O)c1ccc(CSc2nc(=NC3CCNCC3)c3cccn3...
1344,CHEMBL5047887,CHEMBL5083761,COc1ccc(CNc2nc(SCc3ccc(C(=O)Nc4ccccc4N)cc3)nn3...,6.41,COc1ccc(CN=c2[nH]c(SCc3ccc(C(=O)Nc4ccccc4N)cc3...
1345,CHEMBL5047887,CHEMBL3621988,Nc1cc(F)ccc1NC(=O)c1ccc(CNC(=O)/C=C/c2cccnc2)cc1,6.71,Nc1cc(F)ccc1NC(=O)c1ccc(CNC(=O)/C=C/c2cccnc2)cc1
1346,CHEMBL5047887,CHEMBL5083922,COc1ccc(CNc2nc(SCc3ccc(C(=O)Nc4ccc(F)cc4N)cc3)...,5.30,COc1ccc(CN=c2[nH]c(SCc3ccc(C(=O)Nc4ccc(F)cc4N)...


In [35]:
from knn_failure import KNNFailureSplitter

In [36]:
knn_failure_splitter = KNNFailureSplitter(
    similarity_threshold=0.7,
    activity_threshold=0.5,
    n_neighbors=3,
    test_fraction=0.2,
)

In [38]:
knn_failure_splitter.visualise_splits(
    df['standardized_smiles'].values,
    df['pchembl_value'].values,
    intended_biases=np.arange(0, 11) / 10,
    n_repeats=3,
    output_path="knn.gif",
)

![knn](./knn_failure_sweep.gif)